# CFD-Drag-2D: CUDA solver check (M3)

Clones the repository, builds the CPU and CUDA solvers, runs the test suite (including the
GPU tests), compares CPU and GPU results on the same cases, and measures GPU throughput.
Only `summary.json` and the small comparison tables are kept as outputs.

In [ ]:
REPO = "https://github.com/PradeepkrishnaGM/CFD-Drag-2D.git"
BRANCH = "m3-cuda"
WORK = "/kaggle/working"
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!rm -rf /tmp/repo && git clone -q --depth 1 -b $BRANCH $REPO /tmp/repo && git -C /tmp/repo log --oneline -1

In [ ]:
%cd /tmp/repo
!pip install -q -e . --no-deps && pip install -q pytest
import os, subprocess
nvcc_dir = subprocess.run(["bash", "scripts/kaggle_setup_cuda.sh"], capture_output=True, text=True, check=True).stdout.strip().splitlines()[-1]
os.environ["PATH"] = nvcc_dir + ":" + os.environ["PATH"]
!nvcc --version | tail -2
!make -s solver && make -s cuda && ls -la build

In [ ]:
!python -m pytest -q -m "not slow" -rs 2>&1 | tail -15

In [ ]:
!python scripts/compare_backends.py --n 16 --bench 128 --out $WORK/m3

In [ ]:
# keep only small outputs
!find $WORK/m3 -name samples -type d -prune -exec rm -rf {} + ; du -sh $WORK/m3